# Make AI pictures and video with Treza from LangChain

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/treza-labs/treza-examples/blob/main/langchain-colab/treza_langchain_agent.ipynb)

This notebook uses [`langchain-treza`](https://github.com/treza-labs/langchain-treza) to call [Treza](https://www.trezalabs.com).

1. **Call the tools directly.** Check your credit balance and make one of today's free pictures. Costs $0.
2. **Optional: give the tools to an agent.** It makes a 4 second vertical video. About $0.28 on Treza, capped at $0.50, plus what your model provider charges for the agent's tokens.

You need a Treza API key. Make one at https://www.trezalabs.com/platform/api-keys with the `pipelines:read`, `pipelines:write` and `pipelines:run` permissions. Renders are paid from prepaid Treza credits. Every account also gets 5 free pictures a day on the default image model, so part 1 works before you add credits.

In [ ]:
%pip install -q -U langchain-treza

## Your Treza API key

In Colab, open **Secrets** (the key icon on the left), add `TREZA_API_KEY` and turn on notebook access. Anywhere else, set `TREZA_API_KEY` before starting Jupyter, or paste the key when asked. The key is not printed or saved in the notebook.

In [ ]:
import os
from getpass import getpass


def load_key(name: str) -> None:
    """Put an API key in the environment: already set, from Colab's Secrets, or typed in."""
    if os.environ.get(name):
        return
    try:
        from google.colab import userdata  # only in Colab

        value = userdata.get(name)
    except Exception:  # not in Colab, or the secret is missing or not shared with this notebook
        value = None
    os.environ[name] = value or getpass(f"{name}: ")


load_key("TREZA_API_KEY")

## 1. Call the tools directly

Each tool takes a dict of arguments and returns a JSON string. Checking the balance is free.

In [ ]:
import json

from langchain_treza import TrezaBalance

print(json.loads(TrezaBalance().invoke({})))

### A free picture

`max_cost_usd` is the most one render may cost. Treza prices the render first, and the tool starts nothing when the price is higher. One of today's free pictures is priced at $0, so a cap of $0.01 lets a free picture through and stops a paid one (about $0.06 once today's free pictures are used up; they come back at midnight UTC).

In [ ]:
from IPython.display import Image, display

from langchain_treza import TrezaGenerateImage

picture = TrezaGenerateImage(max_cost_usd=0.01)
answer = picture.invoke({"prompt": "A lighthouse on a sea cliff at golden hour, loose watercolor", "aspect_ratio": "16:9"})

if answer.startswith("{"):
    image = json.loads(answer)
    print(f"${image['price_usd']:.2f} on {image.get('model')}: {image['url']}")
    display(Image(url=image["url"], width=640))
else:
    print(answer)  # a refusal (over the cap, a bad key...), worded so you know what to change. Nothing was spent.

## 2. Optional: an agent that makes a video

This part spends credits: a 4 second vertical clip on Treza's default video model (Veo 3.1 Lite) is about $0.28. `max_cost_usd=0.50` on the toolkit is the most any one render may cost. The model cannot see or change it, and it cannot see your Treza key.

The agent also needs a chat model. This uses Claude Sonnet 5.5 through `langchain-anthropic`, with an Anthropic API key in `ANTHROPIC_API_KEY` (Colab Secrets, or typed in when asked). OpenAI works the same way: `pip install langchain-openai`, load `OPENAI_API_KEY` and pass `"openai:<model name>"` to `create_agent` instead.

In [ ]:
%pip install -q -U langchain langchain-anthropic

In [ ]:
from langchain.agents import create_agent

from langchain_treza import TrezaToolkit

load_key("ANTHROPIC_API_KEY")

tools = TrezaToolkit(max_cost_usd=0.50, include_pipelines=False).get_tools()
agent = create_agent(
    "anthropic:claude-sonnet-5-5",
    tools,
    system_prompt="You make pictures and videos with Treza's tools. Make what is asked in one call, then say what you made.",
)

reply = agent.invoke(
    {"messages": [{"role": "user", "content": "Make a 4 second vertical video of a hummingbird drinking from a red flower, in slow motion."}]}
)
print(reply["messages"][-1].text)

In [ ]:
import json

from IPython.display import Video, display

# The tool's own answer has the file's url, so read it there rather than from the agent's reply.
for message in reply["messages"]:
    if message.type != "tool" or message.name != "treza_generate_video":
        continue
    if not message.content.startswith("{"):
        print(message.content)  # a refusal: over the cap, low balance... Nothing was spent.
        continue
    clip = json.loads(message.content)
    if clip.get("url"):
        print(f"${clip['price_usd']:.2f} on {clip.get('model')}: {clip['url']}")
        display(Video(clip["url"], width=360))
    else:
        print(f"Still rendering. Check it later with TrezaGetRun: pipeline_id={clip['pipeline_id']}, run_id={clip['run_id']}")

## Next

- [`langchain-treza` README](https://github.com/treza-labs/langchain-treza): every tool, spending caps, long renders.
- [treza-examples](https://github.com/treza-labs/treza-examples): the same flow from curl, Python, TypeScript, n8n and ComfyUI, and a narrated Short from one prompt.
- `TrezaRunPipeline` runs a pipeline you saved in Treza, with new inputs.